In [18]:
import sys
from pathlib import Path


def project_src(start: Path) -> Path:
    """Return the src directory that contains the tiny_story package."""
    for candidate in (start, *start.parents):
        src = candidate / "src"
        if (src / "tiny_story").is_dir():
            return src
    message = f"tiny_story package not found above {start}"
    raise RuntimeError(message)


SRC = project_src(Path.cwd())
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [19]:
from tiny_story.infrastructure.logging.filesystem_run_logger import FilesystemRunLogger
# 1. Create a log file
LOG_PATH = SRC.parent / "notebooks" / "data" / "check-run.log"
log = FilesystemRunLogger(LOG_PATH)
# 2. Log a message
log.log("test message")
# 3. Close the log file
log.close()
# 4. Read the log file
LOG_PATH.read_text(encoding="utf-8")

2026-10-14 17:17:57.441 | INFO     | tiny_story.infrastructure.logging.filesystem_run_logger:log:60 - test message


'2026-10-14 20:15:37 | INFO | test message\n2026-10-14 15:19:14 | INFO | test message\n2026-10-14 15:37:14 | INFO | test message\n2026-10-14 15:50:37 | INFO | test message\n2026-10-14 16:00:19 | INFO | test message\n2026-10-14 16:30:06 | INFO | test message\n2026-10-14 16:34:09 | INFO | test message\n2026-10-14 17:17:57 | INFO | test message\n'

In [20]:
import torch
import lightning

In [21]:
torch.cuda.is_available()

True

In [22]:
from tiny_story.infrastructure.writers.filesystem_prediction_writer import (
    FilesystemPredictionWriter,
)

# 1. Create a prediction writer
OUTPUT_DIR = SRC.parent / "notebooks" / "data" / "predictions"
writer = FilesystemPredictionWriter(OUTPUT_DIR)
# 2. Write a prediction
writer.write("check", {"label": "ok", "score": 1})
# 3. Read the prediction file
(OUTPUT_DIR / "check.json").read_text(encoding="utf-8")

'{\n  "label": "ok",\n  "score": 1\n}'

In [23]:
from dataclasses import dataclass

from tiny_story.infrastructure.config.yaml_config_repository import (
    YamlConfigRepository,
)


@dataclass(frozen=True)
class CheckConfig:
    """Configuration loaded by this notebook check."""

    name: str
    count: int


def parse_check(raw: dict[str, object]) -> CheckConfig:
    """Build a check configuration from a YAML mapping.

    Args:
        raw: Mapping loaded from YAML.

    Returns:
        The check configuration.
    """
    name = raw["name"]
    count = raw["count"]
    if not isinstance(name, str) or not isinstance(count, int):
        message = "Check YAML values have unexpected types"
        raise TypeError(message)
    return CheckConfig(name=name, count=count)


# 1. Create a config file
CONFIG_PATH = SRC.parent / "notebooks" / "data" / "check.yaml"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text("name: check\ncount: 1\n", encoding="utf-8")
# 2. Create a config repository
repository = YamlConfigRepository(
    {CheckConfig: CONFIG_PATH},
    {CheckConfig: parse_check},
)
# 3. Load the configuration
repository.get(CheckConfig)


CheckConfig(name='check', count=1)